<a href="https://colab.research.google.com/github/vchiang001/Intro2BehvRes/blob/main/ontario_avian_monitor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Assisted Avian Monitoring for Urban Creek Restoration: Automated Species, Pose, and Behavior Analysis

**A multi-modal computer vision demonstration using BioCLIP, DeepLabCut SuperAnimal, and X-CLIP** </br>
Reference Context: Jefferson, Jordan, and Jayfield (JJJ) Parks Naturalization Project, Don Doan Trail Corridor, Brampton, Ontario.  
This notebook executes a multi-modal computer vision workflow to evaluate ecological recovery following urban concrete-channel removal and riparian naturalization. The pipeline automates the extraction of avian community indicators across three primary monitoring axes:</br>
1. Taxonomic identification (BioCLIP): Constraints candidate classifications to regional species occurrence records.   
2. Postural tracking (DeepLabCut SuperAnimal): Extracts multi-individual anatomical keypoints without requiring de novo network training.   
3. Behavioral and habitat classification (X-CLIP): Scores ethological actions and fine-scale habitat associations across riparian restoration zones.

Evaluating habitat restoration along urban stream corridors requires steady wildlife observation. Reviewing field video manually is time-consuming and demands specialized bird knowledge. This system allows anyone to upload raw field recordings. The software extracts biological data directly from the footage without manual frame tagging or labeling.

**Resources used**
1. Ni, B., Peng, H., Chen, M., Zhang, S., Meng, G., Fu, J., Xiang, S., & Ling, H. (2022). Expanding Language-Image Pretrained Models for General Video Recognition. In S. Avidan, G. Brostow, M. Cissé, G. M. Farinella, & T. Hassner (Eds.), Computer Vision – ECCV 2022 (pp. 1–18). Springer Nature Switzerland. https://doi.org/10.1007/978-3-031-19772-7_1
2. Stevens, S., Wu, J., Thompson, M. J., Campolongo, E. G., Song, C. H., Carlyn, D. E., Dong, L., Dahdul, W. M., Stewart, C., Berger-Wolf, T., Chao, W.L., & Su, Y. (2024). BioCLIP: A Vision Foundation Model for the Tree of Life. 2024 IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR), 19412–19424. https://doi.org/10.1109/CVPR52733.2024.01836
3. Ye, S., Filippova, A., Lauer, J., Schneider, S., Vidal, M., Qiu, T., Mathis, A., & Mathis, M. W. (2024). SuperAnimal pretrained pose estimation models for behavioral analysis. Nature Communications, 15(1), 5165. https://doi.org/10.1038/s41467-024-48792-2

<a href="https://drive.google.com/file/d/1Fde8N9mLAxRYa7ust4pKI5jsgViga6-1/view?usp=sharing" target="_blank" rel="noopener noreferrer">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a href="https://huggingface.co/spaces/vsccvscc/ontario_avian_monitor" target="_blank" rel="noopener noreferrer" style="margin-left: 8px;">
  <img src="https://img.shields.io/badge/%F0%9F%A4%97%20Gradio-Open%20in%20Spaces-orange" alt="Open in Gradio"/>
</a>

## Environment Configuration
Configure runtime execution: select ```Runtime → Change runtime type```, set the environment to ```Python 3```, and select ```GPU``` acceleration. </br>

Dependency Installation Note: DeepLabCut and PyTorch video utilities require session initialization. If prompted following package installation, select ```Runtime → Restart session``` prior to loading models.

In [ ]:
# Install the dependencies (this will take a few minutes to install all the dependencies!)
!pip install -q pybioclip deeplabcut dlclibrary transformers decord opencv-python pillow requests

## Configuration & Setup
Let's import our libraries and configure our workspace directories. </br>
Change ```video_dir``` & ```output_dir``` to where your videos are, and where you want the result to be saved. Everything else can be kept the same.

In [ ]:
import os
import itertools
import requests
import cv2
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from decord import VideoReader, cpu
from bioclip import CustomLabelsClassifier
from dlclibrary import download_huggingface_model
import deeplabcut
from transformers import AutoProcessor, XCLIPModel

# --- Configuration ---
video_dir = Path("./data/videos")
video_dir.mkdir(parents=True, exist_ok=True) # Ensure input dir exists so you can upload to it in Colab

output_dir = Path("./data/pipeline_results")
output_dir.mkdir(parents=True, exist_ok=True)

species_report_path = output_dir / "species_identification_summary.txt"
behavior_report_path = output_dir / "avian_behavior_summary.txt"
pose_output_dir = output_dir / "pose_estimation"
pose_output_dir.mkdir(exist_ok=True)

model_cache_dir = Path("./models/superanimal_bird")
model_cache_dir.mkdir(parents=True, exist_ok=True)

## 1. What Bird Is It? Taxonomic Classification via Geographically Constrained Zero-Shot BioCLIP
Open-set visual classifiers are susceptible to geographical misclassification. This module restricts the label candidate space by querying research-grade iNaturalist observation tallies for Ontario (Place ID 6883). Frame sampling across uniform temporal intervals ensures target detection despite temporary foliage occlusions in dense riparian canopy and shrub layers.

In [ ]:
def fetch_ontario_bird_checklist(place_id=6883):
    # Restrict candidate space via iNaturalist to prevent geographic misclassification
    url = "https://api.inaturalist.org/v1/observations/species_counts"
    params = {"place_id": place_id, "taxon_id": 3, "quality_grade": "research", "per_page": 500}
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()

    taxon_map = {}
    bird_names = []
    for item in resp.json().get("results", []):
        taxon = item.get("taxon", {})
        common = taxon.get("preferred_common_name")
        scientific = taxon.get("name")
        if common:
            name = common.title()
            taxon_map[name] = scientific
            bird_names.append(name)
        elif scientific:
            taxon_map[scientific] = scientific
            bird_names.append(scientific)
    return bird_names, taxon_map

def run_species_id(vids, report_path, sample_frames=50, top_k=5):
    bird_names, taxon_map = fetch_ontario_bird_checklist()
    classifier = CustomLabelsClassifier(bird_names)

    with open(report_path, "w", encoding="utf-8") as report:
        for v in vids:
            cap = cv2.VideoCapture(str(v))
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total_frames <= 0:
                cap.release()
                raise ValueError(f"Could not read {v.name}. Check if the video codec is supported.")

            # Sample 50 frames to reliably hit unoccluded foliage gaps
            f_indices = np.linspace(0, total_frames - 1, sample_frames, dtype=int)
            cum_scores = {name: 0.0 for name in bird_names}
            scored_frames = 0

            for i, idx in enumerate(f_indices):
                cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
                ret, frame = cap.read()
                if ret:
                    tmp_img = f"temp_frame_{i}.jpg"
                    Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).save(tmp_img)

                    preds = classifier.predict(tmp_img)
                    for p in preds:
                        cum_scores[p["classification"]] += p["score"]
                    scored_frames += 1

                    if os.path.exists(tmp_img):
                        os.remove(tmp_img)
            cap.release()

            if scored_frames == 0:
                raise ValueError(f"Failed to extract readable frames from {v.name}!")

            mean_scores = {name: score / scored_frames for name, score in cum_scores.items()}
            ranked_preds = sorted(mean_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

            block = f"File: {v.name} ({scored_frames} frames analyzed)\n" + "-" * 55 + "\n"
            for r, (name, score) in enumerate(ranked_preds, start=1):
                sci = taxon_map.get(name, "Unknown")
                block += f"  {r}. {name:<26} [{sci}] : {score * 100:.1f}%\n"
            report.write(block + "\n")
            report.flush()

## 2. How Is the Bird Moving? Markerless Avian Pose Estimation via DeepLabCut SuperAnimal

To quantify postural adjustments and movement patterns without manual annotation, this module loads the pre-trained ```superanimal_bird``` architecture (ResNet-50 backbone with Faster R-CNN detection). Network weights are cached locally to evaluate multi-individual kinematics across unconstrained field sequences.

Quantifying movement requires tracking physical posture over time.   
- Keypoint Tracking: The model (DeepLabCut SuperAnimal-Bird) automatically detects anatomical landmarks—including the beak, eyes, nape, wings, and feet—without requiring user-labeled examples.   
- Video Adaptation: The algorithm refines its detections directly on the video clip to reduce tracking jitter between frames.   
- Connection to Behavior: Tracking changes in body positions—such as head tilt angles during vigilance, wing flutters during territorial defense, or bill trajectories during foraging bouts—provides an objective measurement of action.   
- Current Model Boundaries: Dense brush, low video resolution, and distant subjects can still challenge the pose estimator. In these cases, the pipeline defaults to standard zero-shot detections rather than stopping.

In [ ]:
def download_superanimal_bird_weights(weights_dir):
    # Pre-cache both backbone weights explicitly to avoid DLCLibrary download look-up errors
    download_huggingface_model("superanimal_bird_fasterrcnn_mobilenet_v3_large_fpn", weights_dir)
    download_huggingface_model("superanimal_bird_resnet_50", weights_dir)

def track_bird_poses(vids, dest_dir, max_individuals=3):
    download_superanimal_bird_weights(model_cache_dir)

    for v in vids:
        deeplabcut.video_inference_superanimal(
            videos=[str(v)],
            superanimal_name="superanimal_bird",
            model_name="resnet_50",
            detector_name="fasterrcnn_mobilenet_v3_large_fpn",
            video_adapt=True,
            max_individuals=max_individuals,
            pseudo_threshold=0.1,
            bbox_threshold=0.9,
            detector_epochs=1,
            pose_epochs=1,
            dest_folder=str(dest_dir),
        )

## 3. What Is the Bird Doing and Where? Ethological and Micro-Habitat Classification via Video Transformer (X-CLIP)

Evaluating restoration success requires documenting functional habitat use rather than occurrence alone. This module pairs a catalog of 16 ethological actions (foraging, vocalizing, perching) with 16 habitat substrates (such as emergent channel vegetation, restored riparian thickets, and native tree canopies). Zero-shot video-text matching via ```xclip-base-patch16``` generates structured records of restoration feature utilization.

In [ ]:
def build_ethogram_catalog():
    actions = [
        "perched alert and scanning surroundings",
        "perched stationary, resting, or sunning",
        "foraging, pecking, or scratching on the ground",
        "pecking or probing a tree trunk or bark",
        "eating seeds, suet, or nectar from a bird feeder",
        "preening, oiling, or grooming feathers",
        "singing, calling, or vocalizing with beak open",
        "flying or taking off into the air",
        "landing on a branch or ground surface",
        "hopping or walking along the ground",
        "swimming, paddling, or floating on water",
        "diving or dabbling underwater for food",
        "spreading, stretching, or flicking wings",
        "bathing or shaking off water or dust",
        "chasing, displaying territorially, or interacting aggressively",
        "roosting or sleeping with head tucked"
    ]

    habitats = [
        "in a leafy deciduous tree or shrub canopy",
        "on bare deciduous branches or twigs",
        "in a pine, spruce, or cedar evergreen tree",
        "on snow-covered ground or packed ice",
        "on snow-draped evergreen boughs or branches",
        "at a wooden hopper or platform bird feeder",
        "at a hanging wire cage suet or tube feeder",
        "on an open grassy lawn, park turf, or meadow",
        "on the forest floor among fallen autumn leaves",
        "clinging vertically to rough tree trunk bark",
        "inside a dense thicket, brush pile, or bramble",
        "on open freshwater lake, river, or pond water",
        "along a muddy wetland, marsh edge, or reed bed",
        "on a wooden fence post, deck railing, or pergola",
        "on damp soil, mud puddles, or emerging spring grass",
        "on an asphalt roadway, gravel driveway, or stone path"
    ]

    return [f"a bird {act} {hab}" for act, hab in itertools.product(actions, habitats)]

def run_ethogram_classification(vids, report_path, top_k=5):
    ethogram_queries = build_ethogram_catalog()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_id = "microsoft/xclip-base-patch16-zero-shot"

    processor = AutoProcessor.from_pretrained(model_id)
    model = XCLIPModel.from_pretrained(model_id).to(device)

    with open(report_path, "w", encoding="utf-8") as report:
        for v in vids:
            vr = VideoReader(str(v), ctx=cpu(0))
            if len(vr) <= 0:
                raise ValueError(f"{v.name} has no readable video frames!")

            # X-CLIP zero-shot checkpoint strictly requires an input sequence of 32 frames
            indices = np.linspace(0, len(vr) - 1, 32, dtype=int)
            sampled_frames = [vr[idx].asnumpy() for idx in indices]

            inputs = processor(
                text=ethogram_queries,
                videos=list(sampled_frames),
                return_tensors="pt",
                padding=True
            ).to(device)

            with torch.no_grad():
                outputs = model(**inputs)
                probs = outputs.logits_per_video.softmax(dim=1)[0]

            top_scores, top_indices = torch.topk(probs, k=min(top_k, len(ethogram_queries)))

            block = f"File: {v.name}\n" + "-" * 70 + "\n"
            for rank, (p_idx, score) in enumerate(zip(top_indices, top_scores), start=1):
                block += f"  {rank}. {ethogram_queries[p_idx.item()]:<65} : {score.item() * 100:.1f}%\n"
            report.write(block + "\n")
            report.flush()

## Batch Analysis and Summary Generation

Transfer field recordings (e.g., trail-camera monitoring footage or citizen-science submissions) to the designated ```video_dir```
Execution processes all supported video formats, logging ranked taxonomic probabilities, keypoint coordinates, and behavioral classifications to ```output_dir```

In [ ]:
allowed_exts = {".mp4", ".mov", ".avi", ".mkv"}
video_files = sorted([f for f in video_dir.iterdir() if f.suffix.lower() in allowed_exts])

if not video_files:
    print("No video files found! Please upload files to the './data/videos' folder first.")
else:
    print(f"Found {len(video_files)} videos. Starting pipeline...")

    print("\n--- Running Species Identification ---")
    run_species_id(video_files, species_report_path)

    print("\n--- Running Pose Tracking ---")
    track_bird_poses(video_files, pose_output_dir)

    print("\n--- Running Behavior Recognition ---")
    run_ethogram_classification(video_files, behavior_report_path)

    print(f"\nPipeline Complete! Check the '{output_dir}' folder for your text reports and pose data.")